# Finishing Position Model — EDA & Evaluation
Lightweight check on the training data and model performance.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv('data_finishing.csv')
model_bundle = joblib.load('../models/finishing.pkl')

model      = model_bundle['model']
le_driver  = model_bundle['le_driver']
le_circuit = model_bundle['le_circuit']

print('=== DATA OVERVIEW ===')
print(f'Rows         : {len(df)}')
print(f'Seasons      : {sorted(df["season"].unique())}')
print(f'Circuits     : {df["circuit"].nunique()} unique')
print(f'Drivers      : {df["driver"].nunique()} unique')
print(f'Nulls        : {df.isnull().sum().sum()}')

=== DATA OVERVIEW ===
Rows         : 1938
Seasons      : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Circuits     : 32 unique
Drivers      : 34 unique
Nulls        : 0


In [2]:
# Re-encode and split the same way as training
df = df.dropna()
df['driver_enc']  = le_driver.transform(df['driver'])
df['circuit_enc'] = le_circuit.transform(df['circuit'])

X = df[['driver_enc', 'circuit_enc', 'season', 'grid_position']]
y = df['finishing_position']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_pred = model.predict(X_test)
diff   = np.abs(y_pred - y_test)

print('=== MODEL EVALUATION ===')
print(f'Test rows         : {len(X_test)}')
print(f'Exact accuracy    : {accuracy_score(y_test, y_pred):.2%}')
print(f'Within 1 position : {(diff <= 1).mean():.2%}')
print(f'Within 3 positions: {(diff <= 3).mean():.2%}')
print(f'Within 5 positions: {(diff <= 5).mean():.2%}')
print(f'Mean abs error    : {diff.mean():.2f} positions')

=== MODEL EVALUATION ===
Test rows         : 388
Exact accuracy    : 80.93%
Within 1 position : 85.05%
Within 3 positions: 88.92%
Within 5 positions: 92.27%
Mean abs error    : 1.04 positions


In [3]:
print('=== FEATURE IMPORTANCES ===')
for feat, imp in sorted(zip(X.columns, model.feature_importances_), key=lambda x: -x[1]):
    bar = '█' * int(imp * 40)
    print(f'{feat:<15} {imp:.4f}  {bar}')

=== FEATURE IMPORTANCES ===
grid_position   0.3120  ████████████
circuit_enc     0.2900  ███████████
driver_enc      0.2514  ██████████
season          0.1466  █████


In [4]:
print('=== KNOWN DRIVERS & CIRCUITS ===')
print(f'Drivers  ({len(le_driver.classes_)}):', list(le_driver.classes_))
print()
print(f'Circuits ({len(le_circuit.classes_)}):', list(le_circuit.classes_))

=== KNOWN DRIVERS & CIRCUITS ===
Drivers  (34): ['Alexander Albon', 'Antonio Giovinazzi', 'Brendon Hartley', 'Carlos Sainz', 'Charles Leclerc', 'Daniel Ricciardo', 'Daniil Kvyat', 'Esteban Ocon', 'Fernando Alonso', 'George Russell', 'Guanyu Zhou', 'Jack Aitken', 'Kevin Magnussen', 'Kimi Räikkönen', 'Lance Stroll', 'Lando Norris', 'Lewis Hamilton', 'Marcus Ericsson', 'Max Verstappen', 'Mick Schumacher', 'Nicholas Latifi', 'Nico Hulkenberg', 'Nikita Mazepin', 'Nyck De Vries', 'Pierre Gasly', 'Pietro Fittipaldi', 'Robert Kubica', 'Romain Grosjean', 'Sebastian Vettel', 'Sergey Sirotkin', 'Sergio Perez', 'Stoffel Vandoorne', 'Valtteri Bottas', 'Yuki Tsunoda']

Circuits (32): ['Austin', 'Baku', 'Barcelona', 'Budapest', 'Hockenheim', 'Imola', 'Istanbul', 'Jeddah', 'Le Castellet', 'Lusail', 'Melbourne', 'Mexico City', 'Miami', 'Monaco', 'Monte Carlo', 'Montréal', 'Monza', 'Mugello', 'Nürburgring', 'Portimão', 'Sakhir', 'Shanghai', 'Silverstone', 'Singapore', 'Sochi', 'Spa-Francorchamps', 'Spie

# Lap Time Model — EDA & Evaluation
Ridge Regression predicting lap time (ms) from driver, circuit, compound, lap number, and season.

In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

df_lt = pd.read_csv('data_laptime.csv').dropna()
bundle_lt = joblib.load('../models/laptime.pkl')

model_lt    = bundle_lt['model']
le_drv_lt   = bundle_lt['le_driver']
le_cir_lt   = bundle_lt['le_circuit']
le_comp_lt  = bundle_lt['le_compound']

print('=== DATA OVERVIEW ===')
print(f'Rows      : {len(df_lt)}')
print(f'Seasons   : {sorted(df_lt["season"].unique())}')
print(f'Drivers   : {df_lt["driver"].nunique()} unique')
print(f'Circuits  : {df_lt["circuit"].nunique()} unique')
print(f'Compounds : {sorted(df_lt["compound"].unique())}')
print(f'Lap time  : {df_lt["lap_time_ms"].min()/1000:.3f}s – {df_lt["lap_time_ms"].max()/1000:.3f}s  (mean {df_lt["lap_time_ms"].mean()/1000:.3f}s)')

=== DATA OVERVIEW ===
Rows      : 153903
Seasons   : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Drivers   : 45 unique
Circuits  : 34 unique
Compounds : ['Hard', 'Intermediate', 'Medium', 'Soft', 'Wet']
Lap time  : 60.011s – 149.461s  (mean 88.958s)


In [6]:
# Re-encode using saved encoders (same as training)
df_lt['driver_enc']   = le_drv_lt.transform(df_lt['driver'])
df_lt['circuit_enc']  = le_cir_lt.transform(df_lt['circuit'])
df_lt['compound_enc'] = le_comp_lt.transform(df_lt['compound'])

FEATURES_LT = ['driver_enc', 'circuit_enc', 'compound_enc', 'lap_number', 'season']
X_lt = df_lt[FEATURES_LT]
y_lt = df_lt['lap_time_ms']

X_train_lt, X_test_lt, y_train_lt, y_test_lt = train_test_split(
    X_lt, y_lt, test_size=0.2, random_state=42
)

y_pred_lt = model_lt.predict(X_test_lt)
mae_ms    = mean_absolute_error(y_test_lt, y_pred_lt)
r2        = r2_score(y_test_lt, y_pred_lt)

print('=== MODEL EVALUATION ===')
print(f'Test rows : {len(X_test_lt)}')
print(f'MAE       : {mae_ms:.0f} ms  ({mae_ms/1000:.3f} s)')
print(f'R²        : {r2:.4f}')
print()

# Within-N-second accuracy
diff_s = np.abs(y_pred_lt - y_test_lt) / 1000
for thresh in [0.5, 1.0, 2.0, 3.0]:
    pct = (diff_s <= thresh).mean()
    print(f'Within {thresh:.1f}s : {pct:.2%}')

print()
print('=== RIDGE COEFFICIENTS ===')
for feat, coef in sorted(zip(FEATURES_LT, model_lt.coef_), key=lambda x: -abs(x[1])):
    bar = '█' * int(abs(coef) / max(abs(model_lt.coef_)) * 30)
    sign = '+' if coef >= 0 else '-'
    print(f'{feat:<15} {sign}{abs(coef):8.2f} ms  {bar}')

=== MODEL EVALUATION ===
Test rows : 30781
MAE       : 9033 ms  (9.033 s)
R²        : 0.0958

Within 0.5s : 1.73%
Within 1.0s : 3.73%
Within 2.0s : 8.29%
Within 3.0s : 13.62%

=== RIDGE COEFFICIENTS ===
compound_enc    -  730.68 ms  ██████████████████████████████
season          -  227.40 ms  █████████
circuit_enc     -  174.83 ms  ███████
lap_number      -  170.02 ms  ██████
driver_enc      +    1.59 ms  


# Overtake & Safety Car Model — EDA & Evaluation
Two Logistic Regression pipelines (StandardScaler + LogisticRegression) trained on per-race circuit/weather features.
- **Overtake model**: binary High/Low overtaking activity (middle-third rows excluded at training time)
- **Safety Car model**: binary SC deployed / not deployed

In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

df_ov = pd.read_csv('data_overtake.csv').dropna()
bundle_ov = joblib.load('../models/overtake.pkl')

pipe_ov    = bundle_ov['model_overtake']
pipe_sc    = bundle_ov['model_sc']
le_cir_ov  = bundle_ov['le_circuit']
le_wx_ov   = bundle_ov['le_weather']
FEATURES_OV = bundle_ov['features']

print('=== DATA OVERVIEW ===')
print(f'Rows     : {len(df_ov)}')
print(f'Seasons  : {sorted(df_ov["season"].unique())}')
print(f'Circuits : {df_ov["circuit"].nunique()} unique')
print(f'SC rate  : {df_ov["sc_deployed"].mean()*100:.1f}%  ({int(df_ov["sc_deployed"].sum())} races with SC)')
print(f'avg_pos_change — mean: {df_ov["avg_pos_change"].mean():.2f}  '
      f'std: {df_ov["avg_pos_change"].std():.2f}  '
      f'min: {df_ov["avg_pos_change"].min():.2f}  '
      f'max: {df_ov["avg_pos_change"].max():.2f}')

=== DATA OVERVIEW ===
Rows     : 149
Seasons  : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Circuits : 34 unique
SC rate  : 58.4%  (87 races with SC)
avg_pos_change — mean: 3.52  std: 1.18  min: 0.67  max: 7.42


In [8]:
# Re-encode using saved encoders
df_ov['circuit_enc'] = le_cir_ov.transform(df_ov['circuit'])
df_ov['weather_enc'] = le_wx_ov.transform(df_ov['weather'])

# Build overtake label (same logic as export_model3.py — 40th percentile threshold)
ov_threshold = bundle_ov['ov_threshold']
df_ov['overtake_label'] = (df_ov['avg_pos_change'] >= ov_threshold).astype(int)

X_full = df_ov[FEATURES_OV].copy()
y_ov_full = df_ov['overtake_label']
y_sc_full  = df_ov['sc_deployed']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── OVERTAKE MODEL ────────────────────────────────────────────────────────────
print('=== OVERTAKE MODEL — 5-FOLD CV ===')
cv_acc = cross_val_score(pipe_ov, X_full, y_ov_full, cv=skf, scoring='accuracy')
cv_f1  = cross_val_score(pipe_ov, X_full, y_ov_full, cv=skf, scoring='f1_macro')
cv_rec = cross_val_score(pipe_ov, X_full, y_ov_full, cv=skf, scoring='recall_macro')
print(f'CV Accuracy   : {cv_acc.mean():.3f} ± {cv_acc.std():.3f}')
print(f'CV F1 Macro   : {cv_f1.mean():.3f} ± {cv_f1.std():.3f}')
print(f'CV Recall Mac : {cv_rec.mean():.3f} ± {cv_rec.std():.3f}')

print()
pred_threshold = bundle_ov['ov_pred_threshold']
y_prob_ov = pipe_ov.predict_proba(X_full)[:, 1]
y_tuned   = (y_prob_ov >= pred_threshold).astype(int)
print(f'Full-data report (pred threshold={pred_threshold}):')
print(classification_report(y_ov_full, y_tuned, target_names=['Low Activity', 'High Activity']))

# ── SAFETY CAR MODEL ──────────────────────────────────────────────────────────
print('=== SAFETY CAR MODEL — 5-FOLD CV ===')
cv_acc_sc = cross_val_score(pipe_sc, X_full, y_sc_full, cv=skf, scoring='accuracy')
cv_f1_sc  = cross_val_score(pipe_sc, X_full, y_sc_full, cv=skf, scoring='f1_macro')
print(f'CV Accuracy   : {cv_acc_sc.mean():.3f} ± {cv_acc_sc.std():.3f}')
print(f'CV F1 Macro   : {cv_f1_sc.mean():.3f} ± {cv_f1_sc.std():.3f}')

print()
print('Full-data report:')
print(classification_report(y_sc_full, pipe_sc.predict(X_full), target_names=['No SC', 'SC Deployed']))

=== OVERTAKE MODEL — 5-FOLD CV ===
CV Accuracy   : 0.531 ± 0.079
CV F1 Macro   : 0.521 ± 0.078
CV Recall Mac : 0.528 ± 0.081

Full-data report (pred threshold=0.4):
               precision    recall  f1-score   support

 Low Activity       0.63      0.30      0.41        56
High Activity       0.68      0.89      0.77        93

     accuracy                           0.67       149
    macro avg       0.65      0.60      0.59       149
 weighted avg       0.66      0.67      0.64       149

=== SAFETY CAR MODEL — 5-FOLD CV ===
CV Accuracy   : 0.570 ± 0.073
CV F1 Macro   : 0.569 ± 0.074

Full-data report:
              precision    recall  f1-score   support

       No SC       0.54      0.66      0.59        62
 SC Deployed       0.71      0.60      0.65        87

    accuracy                           0.62       149
   macro avg       0.63      0.63      0.62       149
weighted avg       0.64      0.62      0.63       149



In [9]:
# Feature coefficients for both pipelines
print('=== OVERTAKE MODEL — COEFFICIENTS ===')
clf_ov   = pipe_ov.named_steps['clf']
scale_ov = pipe_ov.named_steps['scaler']
coefs_scaled_ov = clf_ov.coef_[0] * scale_ov.scale_  # un-scale for comparability
for feat, raw_coef in sorted(zip(FEATURES_OV, clf_ov.coef_[0]), key=lambda x: -abs(x[1])):
    bar  = '█' * int(abs(raw_coef) / max(abs(clf_ov.coef_[0])) * 25)
    sign = '+' if raw_coef >= 0 else '-'
    print(f'{feat:<16} {sign}{abs(raw_coef):.4f}  {bar}')

print()
print('=== SAFETY CAR MODEL — COEFFICIENTS ===')
clf_sc = pipe_sc.named_steps['clf']
for feat, raw_coef in sorted(zip(FEATURES_OV, clf_sc.coef_[0]), key=lambda x: -abs(x[1])):
    bar  = '█' * int(abs(raw_coef) / max(abs(clf_sc.coef_[0])) * 25)
    sign = '+' if raw_coef >= 0 else '-'
    print(f'{feat:<16} {sign}{abs(raw_coef):.4f}  {bar}')

=== OVERTAKE MODEL — COEFFICIENTS ===
weather_enc      +0.5162  █████████████████████████
drs_zones        +0.2801  █████████████
circuit_enc      -0.2221  ██████████
rain_frac        -0.2211  ██████████
n_compounds      -0.2038  █████████
wind_speed       +0.1562  ███████
circuit_length   +0.1214  █████
laptime_std_s    -0.0884  ████

=== SAFETY CAR MODEL — COEFFICIENTS ===
rain_frac        +0.8355  █████████████████████████
laptime_std_s    +0.3606  ██████████
weather_enc      -0.2724  ████████
circuit_enc      +0.2617  ███████
circuit_length   +0.1954  █████
drs_zones        +0.1835  █████
n_compounds      +0.1144  ███
wind_speed       -0.0870  ██


# Constructor Championship Model — EDA & Evaluation
Ridge Regression (Pipeline: StandardScaler → Ridge α=10) predicting total season points per constructor.
Uses lag features (prev season points/rank, 2-year trend, pace per race) with LeaveOneGroupOut CV by season — no future data leaks into any fold.

In [10]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

df_cs = pd.read_csv('data_constructor.csv')
bundle_cs = joblib.load('../models/constructor.pkl')

pipe_cs  = bundle_cs['model']
le_cs    = bundle_cs['le_constructor']
FEATURES_CS = bundle_cs['features']

print('=== DATA OVERVIEW ===')
print(f'Rows         : {len(df_cs)}')
print(f'Seasons      : {sorted(df_cs["season"].unique())}')
print(f'Constructors : {sorted(df_cs["constructor"].unique())}')
print()

# ── Reproduce feature engineering (identical to export_model4.py) ─────────────
df_cs = df_cs.sort_values(['constructor', 'season']).reset_index(drop=True)
df_cs['rank'] = df_cs.groupby('season')['total_points'].rank(
    ascending=False, method='min').astype(int)
df_cs['prev_points']      = df_cs.groupby('constructor')['total_points'].shift(1)
df_cs['prev_rank']        = df_cs.groupby('constructor')['rank'].shift(1)
df_cs['prev_rounds']      = df_cs.groupby('constructor')['rounds_counted'].shift(1)
df_cs['prev2_points']     = df_cs.groupby('constructor')['total_points'].shift(2)
df_cs['points_trend']     = df_cs['prev_points'] - df_cs['prev2_points']
df_cs['avg_pts_per_race'] = df_cs['prev_points'] / df_cs['prev_rounds'].replace(0, np.nan)

df_train_cs = df_cs.dropna(subset=[
    'prev_points', 'prev_rank', 'points_trend', 'avg_pts_per_race'
]).copy()
df_train_cs['constructor_enc'] = le_cs.transform(df_train_cs['constructor'])

X_cs     = df_train_cs[FEATURES_CS].values
y_cs     = df_train_cs['total_points'].values
groups   = df_train_cs['season'].values

print(f'Training rows : {len(df_train_cs)}')
print(f'Season range  : {df_train_cs["season"].min()} – {df_train_cs["season"].max()}')
print()
print(df_train_cs.sort_values(['season','total_points'], ascending=[True,False])
      [['season','constructor','total_points','rank','prev_points','points_trend']]
      .to_string(index=False))

=== DATA OVERVIEW ===
Rows         : 90
Seasons      : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]
Constructors : ['Alfa Romeo', 'AlphaTauri', 'Alpine', 'Aston Martin', 'Ferrari', 'Haas', 'McLaren', 'Mercedes', 'Red Bull', 'Williams']

Training rows : 70
Season range  : 2020 – 2026

 season  constructor  total_points  rank  prev_points  points_trend
   2020     Mercedes         573.0     1        739.0          84.0
   2020     Red Bull         319.0     2        417.0          -2.0
   2020 Aston Martin         210.0     3         73.0         -38.0
   2020      McLaren         202.0     4        145.0          83.0
   2020       Alpine         181.0     5         91.0         -31.0
   2020      Ferrari         131.0     6        504.0         -67.0
   2020   AlphaTauri         107.0     7         85.0          52.0
   2020   Alfa Romeo           8.0     8         57.0           9.0
   

In [11]:
# ── LeaveOneGroupOut CV (season-wise — same as training script) ───────────────
logo   = LeaveOneGroupOut()
cv_mae = cross_val_score(pipe_cs, X_cs, y_cs, groups=groups,
                         cv=logo, scoring='neg_mean_absolute_error')
cv_r2  = cross_val_score(pipe_cs, X_cs, y_cs, groups=groups,
                         cv=logo, scoring='r2')

print('=== MODEL EVALUATION — LeaveOneGroupOut (season-wise) ===')
print(f'MAE  : {-cv_mae.mean():.1f} ± {cv_mae.std():.1f} pts')
print(f'R²   : {cv_r2.mean():.3f}  ± {cv_r2.std():.3f}')

unique_seasons = sorted(set(groups))
print()
print(f'{"Season":<8} {"MAE":>8} {"R²":>8}')
print('-' * 28)
for season, mae_val, r2_val in zip(unique_seasons, [-v for v in cv_mae], cv_r2):
    flag = '  ⚠' if mae_val > 80 else ''
    print(f'{season:<8} {mae_val:>8.1f} {r2_val:>8.3f}{flag}')

# ── Full-data fit ─────────────────────────────────────────────────────────────
y_pred_cs = pipe_cs.predict(X_cs)
print()
print('=== FULL-DATA FIT (optimistic upper bound) ===')
print(f'MAE : {mean_absolute_error(y_cs, y_pred_cs):.1f} pts')
print(f'R²  : {r2_score(y_cs, y_pred_cs):.3f}')

# ── Ridge coefficients ────────────────────────────────────────────────────────
print()
print('=== RIDGE COEFFICIENTS ===')
ridge   = pipe_cs.named_steps['ridge']
scaler  = pipe_cs.named_steps['scaler']
for feat, coef in sorted(zip(FEATURES_CS, ridge.coef_), key=lambda x: -abs(x[1])):
    bar  = '█' * int(abs(coef) / max(abs(ridge.coef_)) * 30)
    sign = '+' if coef >= 0 else '-'
    print(f'{feat:<20} {sign}{abs(coef):8.3f}  {bar}')

=== MODEL EVALUATION — LeaveOneGroupOut (season-wise) ===
MAE  : 100.1 ± 25.4 pts
R²   : -2.351  ± 7.424

Season        MAE       R²
----------------------------
2020         64.5    0.644
2021        106.1    0.577  ⚠
2022         98.2    0.711  ⚠
2023         83.9    0.796  ⚠
2024        124.7    0.596  ⚠
2025         79.2    0.754
2026        143.9  -20.536  ⚠

=== FULL-DATA FIT (optimistic upper bound) ===
MAE : 90.7 pts
R²  : 0.698

=== RIDGE COEFFICIENTS ===
avg_pts_per_race     +  60.326  ██████████████████████████████
rounds_counted       +  58.047  ████████████████████████████
prev_rank            -  52.825  ██████████████████████████
prev_points          +  42.510  █████████████████████
constructor_enc      +  21.967  ██████████
season               -   7.440  ███
points_trend         +   6.447  ███


In [12]:
# ── Latest season inference baseline (what the app uses for predictions) ──────
print('=== INFERENCE BASELINE (latest known stats per constructor) ===')
print(f'Latest season in data: {bundle_cs["latest_season"]}')
print()
print(f'{"Constructor":<16} {"last_pts":>9} {"last_rank":>10} {"last_rounds":>12} {"prev2_pts":>10}')
print('-' * 62)
for team, stats in sorted(bundle_cs['latest_stats'].items(),
                           key=lambda x: x[1].get('last_rank', 99)):
    p2 = stats.get('prev2_points', float('nan'))
    p2_str = f'{p2:.0f}' if not (isinstance(p2, float) and np.isnan(p2)) else 'N/A'
    print(f'{team:<16} {stats["last_points"]:>9.0f} {stats["last_rank"]:>10} '
          f'{stats["last_rounds"]:>12} {p2_str:>10}')

=== INFERENCE BASELINE (latest known stats per constructor) ===
Latest season in data: 2026

Constructor       last_pts  last_rank  last_rounds  prev2_pts
--------------------------------------------------------------
Mercedes               123          1            3        424
Ferrari                 77          2            3        360
McLaren                 38          3            3        775
Haas                    17          4            3         73
Alpine                  16          5            3         20
Red Bull                16          5            3        410
Williams                 2          7            3        124
Alfa Romeo               0          8            3         70
AlphaTauri               0          8            3          0
Aston Martin             0          8            3         80
